In [2]:
# ==========================================
# Import Library
# ==========================================

import os
import cv2
import shutil

import numpy as np
import pandas as pd

from pathlib import Path
from tqdm import tqdm

import matplotlib.pyplot as plt

In [3]:
# ==========================================
# Configuration
# ==========================================

OUTPUT_DIR = Path(r"C:\LipReadingSSL\output")

# ------------------------------------------
# Clip Configuration
# ------------------------------------------

CLIP_LENGTH = 29

STRIDE = 1

IMAGE_SIZE = 96

# ------------------------------------------
# Save Option
# ------------------------------------------

SAVE_JPG = True

SAVE_NPY = True

# RGB / GRAY

COLOR_MODE = "GRAY"

# ------------------------------------------
# Others
# ------------------------------------------

DROP_LAST = True

SHOW_PREVIEW = True

print("=" * 60)
print("Clip Generation Configuration")
print("=" * 60)

print(f"Output Directory : {OUTPUT_DIR}")
print(f"Clip Length      : {CLIP_LENGTH}")
print(f"Stride           : {STRIDE}")
print(f"Image Size       : {IMAGE_SIZE}")
print(f"Save JPG         : {SAVE_JPG}")
print(f"Save NPY         : {SAVE_NPY}")
print(f"Color Mode       : {COLOR_MODE}")

Clip Generation Configuration
Output Directory : C:\LipReadingSSL\output
Clip Length      : 29
Stride           : 1
Image Size       : 96
Save JPG         : True
Save NPY         : True
Color Mode       : GRAY


In [4]:
# ==========================================
# Helper Functions
# ==========================================

def create_output(video):

    out = OUTPUT_DIR / video

    folders = [

        "clips",

        "clip_npy",

        "clip_preview"

    ]

    for folder in folders:

        (out/folder).mkdir(

            parents=True,

            exist_ok=True

        )

    return out


# ------------------------------------------
# Load Mouth Images
# ------------------------------------------

def load_images(video):

    folder = (

        OUTPUT_DIR /

        video /

        "mouth_crop"

    )

    return sorted(

        folder.glob("*.jpg")

    )


# ------------------------------------------
# Load Metadata
# ------------------------------------------

def load_metadata(video):

    path = (

        OUTPUT_DIR /

        video /

        "mouth_metadata.csv"

    )

    if not path.exists():

        return pd.DataFrame()

    return pd.read_csv(path)


# ------------------------------------------
# Clip Name
# ------------------------------------------

def clip_name(index):

    return f"clip_{index:06d}"


# ------------------------------------------
# Save CSV
# ------------------------------------------

def save_csv(df, path):

    df.to_csv(

        path,

        index=False,

        encoding="utf-8-sig"

    )

In [5]:
# ==========================================
# Load Frame Sequence
# ==========================================

def load_frame_sequence(video):

    images = load_images(video)

    if len(images) == 0:

        return []

    frames = []

    for img_path in images:

        try:

            frame_id = int(img_path.stem)

        except:

            frame_id = len(frames)

        frames.append({

            "frame": frame_id,

            "path": img_path

        })

    frames = sorted(

        frames,

        key=lambda x: x["frame"]

    )

    return frames

In [6]:
# ==========================================
# Generate Sliding Window
# Production Version
# ==========================================

def generate_clips(frames):
    
    clips = []

    total = len(frames)

    print(f"Total available frames : {total}")

    if total < CLIP_LENGTH:

        print(
            f"Not enough frames "
            f"({total} < {CLIP_LENGTH})"
        )

        return clips

    for start in range(

        0,

        total - CLIP_LENGTH + 1,

        STRIDE

    ):

        window = frames[

            start :

            start + CLIP_LENGTH

        ]

        clips.append({

            "start": window[0]["frame"],

            "end": window[-1]["frame"],

            "frames": window,

            "num_frames": len(window)

        })

    print(f"Generated clips : {len(clips)}")

    return clips

In [7]:
# ==========================================
# Save Clip
# ==========================================

def save_clip(

    out,

    clip,

    clip_index

):

    clip_folder = (

        out /

        "clips" /

        clip_name(

            clip_index

        )

    )

    if SAVE_JPG:

        clip_folder.mkdir(

            parents=True,

            exist_ok=True

        )

    clip_images = []

    for frame in clip["frames"]:

        img = cv2.imread(

            str(frame["path"])

        )

        if img is None:

            continue

        if COLOR_MODE.upper()=="GRAY":

            img = cv2.cvtColor(

                img,

                cv2.COLOR_BGR2GRAY

            )

        else:

            img = cv2.cvtColor(

                img,

                cv2.COLOR_BGR2RGB

            )

        if img.shape[:2]!=(IMAGE_SIZE,IMAGE_SIZE):

            img = cv2.resize(

                img,

                (IMAGE_SIZE,IMAGE_SIZE)

            )

        clip_images.append(img)

        if SAVE_JPG:

            shutil.copy2(

                frame["path"],

                clip_folder/

                frame["path"].name

            )

    clip_array=np.stack(

        clip_images,

        axis=0

    )

    npy_path=(

        out/

        "clip_npy"/

        f"{clip_name(clip_index)}.npy"

    )

    if SAVE_NPY:

        np.save(

            npy_path,

            clip_array

        )

    return clip_folder,npy_path

In [10]:
# ==========================================
# Preview Clip
# ==========================================

def preview_clip(

    clip,

    out,

    clip_index

):

    if not SHOW_PREVIEW:

        return

    frames = clip["frames"]

    if len(frames) == 0:

        return

    ids = [

        0,

        len(frames)//2,

        len(frames)-1

    ]

    fig, ax = plt.subplots(

        1,

        3,

        figsize=(10,4)

    )

    for i, idx in enumerate(ids):

        img = cv2.imread(

            str(

                frames[idx]["path"]

            )

        )

        img = cv2.cvtColor(

            img,

            cv2.COLOR_BGR2RGB

        )

        ax[i].imshow(img)

        ax[i].set_title(

            f'{frames[idx]["frame"]}'

        )

        ax[i].axis("off")

    plt.tight_layout()

    fig.savefig(

        out /

        "clip_preview" /

        f"{clip_name(clip_index)}.jpg",

        dpi=200

    )

    plt.close(fig)

In [11]:
# ==========================================
# Process Video
# ==========================================

def process_video(video):

    print(f"\nProcessing : {video}")

    out = create_output(video)

    frames = load_frame_sequence(video)

    if len(frames) == 0:

        print("No mouth images found.")

        return []

    clips = generate_clips(frames)

    if len(clips) == 0:

        print("No valid clips generated.")

        return []

    metadata = []

    for idx, clip in enumerate(
        tqdm(clips),
        start=1
    ):

        folder, npy_path = save_clip(
            out,
            clip,
            idx
        )

        if SHOW_PREVIEW and idx <= 10:

            preview_clip(
                clip,
                out,
                idx
            )

        metadata.append({

            "clip_id": clip_name(idx),

            "video": video,

            "jpg_folder": str(
                folder.relative_to(OUTPUT_DIR)
            ),

            "npy_path": str(
                npy_path.relative_to(OUTPUT_DIR)
            ),

            "start_frame": clip["start"],

            "end_frame": clip["end"],

            "num_frames": len(
                clip["frames"]
            ),

            "clip_length": CLIP_LENGTH,

            "stride": STRIDE,

            "color_mode": COLOR_MODE

        })

    print(f"Generated {len(metadata)} clips.")

    return metadata

In [12]:
# ==========================================
# Save Clip Metadata
# ==========================================

def save_clip_metadata(

    video,

    metadata

):

    out = OUTPUT_DIR / video

    df = pd.DataFrame(metadata)

    save_csv(

        df,

        out /

        "clip_metadata.csv"

    )

    return df

In [13]:
# ==========================================
# Run All Videos
# ==========================================

videos = sorted(

    [

        x.name

        for x in OUTPUT_DIR.glob("video*")

        if x.is_dir()

    ]

)

print(f"Found {len(videos)} videos\n")

summary = []

for video in videos:

    metadata = process_video(video)

    df = save_clip_metadata(

        video,

        metadata

    )

    summary.append({

        "video": video,

        "clips": len(df),

        "frames":

            df["num_frames"].sum()

            if len(df)

            else 0

    })

Found 2 videos


Processing : video001
Total available frames : 43021
Generated clips : 42993


100%|██████████| 42993/42993 [1:03:43<00:00, 11.25it/s]


Generated 42993 clips.

Processing : video002
Total available frames : 38888
Generated clips : 38860


100%|██████████| 38860/38860 [57:10<00:00, 11.33it/s] 


Generated 38860 clips.


In [14]:
# ==========================================
# Summary
# ==========================================

summary_df = pd.DataFrame(summary)

display(summary_df)

save_csv(

    summary_df,

    OUTPUT_DIR/

    "clip_summary.csv"

)

print("\n"+"="*60)

print("Phase 6 Finished")

print("="*60)

print(

    f"Videos : "

    f"{len(summary_df)}"

)

print(

    f"Total Clips : "

    f"{summary_df.clips.sum()}"

)

print(

    f"Total Clip Frames : "

    f"{summary_df.frames.sum()}"

)

print("\nOutput Folder")

print(OUTPUT_DIR)

print("\nNext")

print("07_dataset_builder.ipynb")

,video,clips,frames
0,video001,42993,1246797
1,video002,38860,1126940



Phase 6 Finished
Videos : 2
Total Clips : 81853
Total Clip Frames : 2373737

Output Folder
C:\LipReadingSSL\output

Next
07_dataset_builder.ipynb


In [14]:
frames = load_frame_sequence("video002")

print("Total frames :", len(frames))

frame_ids = [x["frame"] for x in frames]

print("First 20 :", frame_ids[:20])

print("Unique diff :", np.unique(np.diff(frame_ids)))

Total frames : 38888
First 20 : [0, 12, 24, 36, 48, 510, 612, 715, 816, 918, 1020, 1122, 1224, 1326, 1429, 1530, 1632, 1734, 1836, 1938]
Unique diff : [        1         2         3         4         5        12       101
       102       103       462      1001      1002      1003      1004
      2002     10001     10002     10003     10004     10005     10006
     10007     10008     20001     20002     20003     20004     20005
     20006     20007     20009     20012     30001     30002     30003
     30005     30006     30007     40001     45102     50009     50019
     60001     80016     80023     90013    100001    100002    100003
    100004    100005    100006    100007    100008    100009    100010
    100011    100012    110001    120018    190040    200001    200002
    200003    200004    200005    200006    200007    200008    200010
    250001    290004    300001    300002    300003    300004    300005
    300006    300009    320035    400001    400003    400006    4000